# Market Data

## Download the Data

- **Purpose:** Download and normalize the fixed-universe trade stream used to build downstream dollar bars.
- **Settings:** `universe=S&P500`; `feed=SIP`; regular sessions in 2025; `MAX_WORKERS=500`.
- **Data:** The local fixed-universe CSV supplies the symbol list; observed trades produce the normalized Parquet cache.
- **Findings:**
- **Decision:** Reuse a matching cache and require deliberate cache replacement to refresh the period.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from src.preprocessing.market_data import ResearchPaths, load_manifest, read_raw

PROJECT_ROOT = Path.cwd().resolve().parents[1]
import pandas as pd
PERIOD = "2025-02-01_2025-12-31"
MANIFEST_PATH = PROJECT_ROOT / "data/preprocessing/universe/sp500_2025.csv"
EXPECTED_SECURITIES = 503
DATA_START = pd.Timestamp("2025-01-01", tz="UTC")
END = pd.Timestamp("2026-01-01", tz="UTC")
paths = ResearchPaths(PROJECT_ROOT, period=PERIOD)
DISPLAY_SYMBOL = "AAPL"
MAX_WORKERS = 500

from src.preprocessing.market_data import collect_raw

universe = load_manifest(MANIFEST_PATH, expected_securities=EXPECTED_SECURITIES)
display(collect_raw(
    paths,
    "tick",
    max_workers=MAX_WORKERS,
    manifest_path=MANIFEST_PATH,
    expected_securities=EXPECTED_SECURITIES,
    start=DATA_START,
    end=END,
))
market_data = read_raw(paths, DISPLAY_SYMBOL, "tick", start=DATA_START, end=END)
market_path = paths.raw(DISPLAY_SYMBOL, "tick")
market_path

## Take a Quick Look at the Data Structure

- **Purpose:** Inspect example rows, dtypes, symbol coverage, and plausible price and trade-size ranges.
- **Settings:**
- **Data:** Inspect cached AAPL ticks; collection covers the full universe and SPY.
- **Findings:**
- **Decision:**

In [ ]:
market_data.head()

In [ ]:
market_data.info()

In [ ]:
market_data["symbol"].value_counts(dropna=False)

In [ ]:
market_data[["price", "size"]].describe()

In [ ]:
market_data[["price", "size"]].hist(bins=50, figsize=(10, 4))
plt.tight_layout()
plt.show()